# NHANES Mortality × Liver Fibrosis — Data Download & Preparation

This notebook downloads NHANES public-use data and linked mortality files for three
cohorts (2007-2008, 2011-2012, 2017-2018), merges the components, and saves
per-cohort parquet files for downstream analysis.

**Data sources:**
- NHANES SAS transport files (demographics, labs, body measures, blood pressure, lipids, glucose, smoking, elastography)
- NCHS Public-Use Linked Mortality Files (2019 release, follow-up through Dec 31, 2019)

In [1]:
import os, hashlib
import requests
import numpy as np
import pandas as pd
import pyreadstat

BASE_DIR = os.path.abspath('.')
RAW_NHANES = os.path.join(BASE_DIR, 'data', 'raw', 'nhanes')
RAW_LMF    = os.path.join(BASE_DIR, 'data', 'raw', 'lmf')
DERIVED    = os.path.join(BASE_DIR, 'data', 'derived')
for d in [RAW_NHANES, RAW_LMF, DERIVED]:
    os.makedirs(d, exist_ok=True)

## Configuration

In [2]:
NHANES_BASE = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public'
LMF_BASE = 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality'

COHORTS = {
    '2007-2008': {
        'year': 2007, 'suffix': 'E',
        'lmf': f'{LMF_BASE}/NHANES_2007_2008_MORT_2019_PUBLIC.dat',
        'has_elast': False,
    },
    '2011-2012': {
        'year': 2011, 'suffix': 'G',
        'lmf': f'{LMF_BASE}/NHANES_2011_2012_MORT_2019_PUBLIC.dat',
        'has_elast': False,
    },
    '2017-2018': {
        'year': 2017, 'suffix': 'J',
        'lmf': f'{LMF_BASE}/NHANES_2017_2018_MORT_2019_PUBLIC.dat',
        'has_elast': True,
    },
}

NHANES_FILES = ['DEMO', 'BIOPRO', 'CBC', 'BMX', 'BPX', 'TRIGLY', 'GLU', 'SMQ']

## Download helpers

In [3]:
def download(url, dest):
    """Download with caching."""
    if os.path.exists(dest):
        return dest
    print(f'  GET {url}')
    r = requests.get(url, timeout=180); r.raise_for_status()
    with open(dest, 'wb') as f: f.write(r.content)
    print(f'    -> {os.path.basename(dest)} ({len(r.content):,} bytes, '
          f'MD5 {hashlib.md5(r.content).hexdigest()})')
    return dest

def read_xpt(path):
    try:
        df, _ = pyreadstat.read_xport(path)
    except UnicodeDecodeError:
        df, _ = pyreadstat.read_xport(path, encoding='latin1')
    return df

def _safe(s, to_type=int):
    s = s.strip().replace('.', '')
    if not s: return None
    try: return to_type(s)
    except ValueError: return None

def parse_lmf(path):
    rows = []
    with open(path) as f:
        for line in f:
            p = line.rstrip('\r\n').ljust(48)
            rows.append({
                'SEQN': _safe(p[0:14]),
                'ELIGSTAT':    _safe(p[14:15]),
                'MORTSTAT':    _safe(p[15:16]),
                'UCOD_LEADING':_safe(p[16:19]),
                'PERMTH_INT':  _safe(p[42:45], float),
                'PERMTH_EXM':  _safe(p[45:48], float),
            })
    return pd.DataFrame(rows)

## Download NHANES + LMF

In [4]:
raw = {}

for cycle, cfg in COHORTS.items():
    print(f'\n=== {cycle} ===')
    yr, sfx = cfg['year'], cfg['suffix']
    cdir = os.path.join(RAW_NHANES, cycle.replace('-','_'))
    os.makedirs(cdir, exist_ok=True)
    d = {}
    
    files_to_get = NHANES_FILES.copy()
    if cfg['has_elast']:
        files_to_get.append('LUX')
    for name in files_to_get:
        fname = f'{name}_{sfx}.xpt'
        url = f'{NHANES_BASE}/{yr}/DataFiles/{fname}'
        path = download(url, os.path.join(cdir, fname))
        d[name] = read_xpt(path)
        print(f'  {name}: {len(d[name]):,} rows')
    
    lmf_path = download(cfg['lmf'],
                        os.path.join(RAW_LMF, os.path.basename(cfg['lmf'])))
    d['LMF'] = parse_lmf(lmf_path)
    print(f'  LMF: {len(d["LMF"]):,} rows')
    
    raw[cycle] = d


=== 2007-2008 ===


  DEMO: 10,149 rows
  BIOPRO: 6,917 rows


  CBC: 9,307 rows
  BMX: 9,762 rows


  BPX: 9,762 rows
  TRIGLY: 3,315 rows
  GLU: 3,315 rows
  SMQ: 7,145 rows
  LMF: 10,149 rows

=== 2011-2012 ===


  DEMO: 9,756 rows
  BIOPRO: 6,549 rows


...

  BPX: 9,338 rows
  TRIGLY: 3,239 rows
  GLU: 3,239 rows
  SMQ: 6,790 rows
  LMF: 9,756 rows

=== 2017-2018 ===


  DEMO: 9,254 rows
  BIOPRO: 6,401 rows


  CBC: 8,366 rows
  BMX: 8,704 rows
  BPX: 8,704 rows


  TRIGLY: 3,036 rows
  GLU: 3,036 rows
  SMQ: 6,724 rows
  LUX: 6,401 rows


  LMF: 9,254 rows


## Merge components into analytic datasets

In [5]:
def merge_cohort(cycle, raw_d, has_elast):
    """Merge all components for one cohort."""
    df = raw_d['DEMO'].copy()
    
    # Labs: AST, ALT from BIOPRO; Platelets from CBC
    bio = raw_d['BIOPRO'][['SEQN','LBXSASSI','LBXSATSI']].rename(
        columns={'LBXSASSI':'AST','LBXSATSI':'ALT'})
    cbc = raw_d['CBC'][['SEQN','LBXPLTSI']].rename(columns={'LBXPLTSI':'PLATELETS'})
    df = df.merge(bio, on='SEQN', how='left').merge(cbc, on='SEQN', how='left')
    
    # BMI
    bmx = raw_d['BMX'][['SEQN','BMXBMI']]
    df = df.merge(bmx, on='SEQN', how='left')
    
    # Blood pressure: average of available systolic readings
    bpx = raw_d['BPX'][['SEQN'] + [c for c in raw_d['BPX'].columns
                                    if c.startswith('BPXSY')]].copy()
    sy_cols = [c for c in bpx.columns if c.startswith('BPXSY')]
    bpx['SBP_MEAN'] = bpx[sy_cols].mean(axis=1)
    df = df.merge(bpx[['SEQN','SBP_MEAN']], on='SEQN', how='left')
    
    # LDL-C (fasting subsample)
    if 'LBDLDL' in raw_d['TRIGLY'].columns:
        df = df.merge(raw_d['TRIGLY'][['SEQN','LBDLDL']], on='SEQN', how='left')
    else:
        df['LBDLDL'] = np.nan
    
    # Fasting plasma glucose
    if 'LBXGLU' in raw_d['GLU'].columns:
        df = df.merge(raw_d['GLU'][['SEQN','LBXGLU']], on='SEQN', how='left')
    else:
        df['LBXGLU'] = np.nan
    
    # Smoking: SMQ020=1 -> ever smoked 100+ cigarettes
    smq = raw_d['SMQ'][['SEQN','SMQ020']].copy()
    smq['SMOKE_EVER'] = (smq['SMQ020'] == 1).astype(float)
    smq.loc[~smq['SMQ020'].isin([1,2]), 'SMOKE_EVER'] = np.nan
    df = df.merge(smq[['SEQN','SMOKE_EVER']], on='SEQN', how='left')
    
    # Elastography
    if has_elast and 'LUX' in raw_d:
        lux = raw_d['LUX'][['SEQN','LUXSMED','LUXCAPM']].rename(
            columns={'LUXSMED':'LSM_KPA','LUXCAPM':'CAP_DBM'})
        df = df.merge(lux, on='SEQN', how='left')
    
    # Linked Mortality
    df = df.merge(raw_d['LMF'], on='SEQN', how='left')
    
    # Filter: eligible adults >= 18
    n0 = len(df)
    df = df[df['ELIGSTAT'] == 1].copy()
    df['AGE'] = df['RIDAGEYR']
    df = df[df['AGE'] >= 18].copy()
    df['FEMALE'] = (df['RIAGENDR'] == 2).astype(float)
    df['CYCLE'] = cycle
    print(f'{cycle}: {n0} -> {len(df)} eligible adults')
    return df

In [6]:
for cycle, cfg in COHORTS.items():
    df = merge_cohort(cycle, raw[cycle], cfg['has_elast'])
    out_path = os.path.join(DERIVED, f'{cycle.replace("-","_")}.parquet')
    df.to_parquet(out_path, index=False)
    print(f'  Saved {out_path} ({os.path.getsize(out_path)/1e6:.1f} MB, {len(df)} rows, {len(df.columns)} cols)')

print('\nDone. Parquet files ready in data/derived/')

2007-2008: 10149 -> 6219 eligible adults
  Saved /home/user/ai_assisted_research/nhanes_mortality_fibrosis/data/derived/2007_2008.parquet (0.3 MB, 6219 rows, 59 cols)


2011-2012: 9756 -> 5849 eligible adults
  Saved /home/user/ai_assisted_research/nhanes_mortality_fibrosis/data/derived/2011_2012.parquet (0.3 MB, 5849 rows, 64 cols)
2017-2018: 9254 -> 5809 eligible adults


  Saved /home/user/ai_assisted_research/nhanes_mortality_fibrosis/data/derived/2017_2018.parquet (0.3 MB, 5809 rows, 64 cols)

Done. Parquet files ready in data/derived/


In [7]:
# Quick sanity check: reload and show shapes
for f in sorted(os.listdir(DERIVED)):
    if f.endswith('.parquet'):
        df = pd.read_parquet(os.path.join(DERIVED, f))
        print(f'{f}: {df.shape[0]} rows × {df.shape[1]} cols')
        print(f'  Columns: {list(df.columns)}')
        print(f'  PERMTH_EXM range: {df["PERMTH_EXM"].min():.0f}–{df["PERMTH_EXM"].max():.0f}')
        print(f'  MORTSTAT=1: {(df["MORTSTAT"]==1).sum()}')
        print()

2007_2008.parquet: 6219 rows × 59 cols
  Columns: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIDEXMON', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDAGEEX', 'RIDRETH1', 'DMQMILIT', 'DMDBORN2', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC3', 'DMDEDUC2', 'DMDSCHOL', 'DMDMARTL', 'DMDHHSIZ', 'DMDFMSIZ', 'INDHHIN2', 'INDFMIN2', 'INDFMPIR', 'RIDEXPRG', 'DMDHRGND', 'DMDHRAGE', 'DMDHRBR2', 'DMDHREDU', 'DMDHRMAR', 'DMDHSEDU', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY', 'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANG', 'WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA', 'AST', 'ALT', 'PLATELETS', 'BMXBMI', 'SBP_MEAN', 'LBDLDL', 'LBXGLU', 'SMOKE_EVER', 'ELIGSTAT', 'MORTSTAT', 'UCOD_LEADING', 'PERMTH_INT', 'PERMTH_EXM', 'AGE', 'FEMALE', 'CYCLE']
  PERMTH_EXM range: 0–159
  MORTSTAT=1: 1126

2011_2012.parquet: 5849 rows × 64 cols
  Columns: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGY', 'RIDEXAGM', 'DMQMILIZ', 'DMQADFC', 'DMDBORN4', 'DMDC